<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/05-skills-and-progressive-disclosure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skills & Progressive Disclosure (concept)

**Goal:** Understand how to give an agent deep, reusable know-how *without* paying for it in context on every turn — the **progressive-disclosure** pattern — and know its concrete instance, the **Agent Skill** (`SKILL.md`), well enough to reach for it and defend the choice.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **This is a concept notebook — no code to run.** Like [MCP](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/04-mcp-and-the-tool-ecosystem.ipynb), Skills are a *packaging + runtime* layer that lives inside an agent host (Claude Code, the Claude apps, the Agent SDK, the Messages API's code-execution container) — not something you invoke against a raw `chat.completions` call on Groq. So we teach the **durable pattern** and show the real `SKILL.md` shape as reference, the same way we did for MCP. You built the tool loop by hand in `01`–`03`; this is the other half of the ecosystem story.

## The problem: context is a budget

By now you've felt the squeeze from both sides of the context window:

- In [`01-model-apis/04-context-and-caching`](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/01-model-apis/04-context-and-caching.ipynb) you saw that every token in the prompt costs money and latency, *every call*.
- In [`02-tool-design`](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/02-tool-design.ipynb) you saw that every tool schema you hand the model sits in context permanently, and that too many bloats the prompt and confuses tool selection.

Now imagine the real job. You want your agent to be genuinely good at a dozen specialized tasks — filling a specific PDF form, generating a branded PowerPoint, following your company's incident-runbook, formatting a financial report to spec. Each of those needs *paragraphs* of instructions, maybe example files, maybe a validated script. 

The naive move is to stuff all of it into the system prompt. That fails on arithmetic: a dozen detailed playbooks is tens of thousands of tokens the model carries — and pays for — on **every single turn**, even when the current task touches none of them. And a prompt that tries to be an expert at everything is, in practice, reliably an expert at nothing.

> **🚩 Common mistake —** treating the system prompt as unlimited storage for "everything the agent might ever need to know." Context is a budget you spend every call; procedural knowledge the agent needs *occasionally* shouldn't be billed *constantly*.

So the real question this notebook answers: **how do you give an agent broad, deep capability while only paying context for the sliver it's using right now?**

## The durable pattern: progressive disclosure

The answer is an idea worth carrying to any agent stack, not just one vendor's: **separate the *what* from the *when/how*, and load the detail on demand.**

Instead of one giant always-resident prompt, you package each capability as a self-contained unit with a cheap **label** (a name + a one-line description of *what it does and when to use it*) and an expensive **body** (the full instructions, examples, files, scripts). Only the labels stay in context. The model reads the body of a unit *only when a request matches its description* — and reads bundled files or runs bundled scripts only if the body actually calls for them.

That's **progressive disclosure**. It decouples two things people usually conflate:

- the **breadth of capability** an agent advertises (can be huge), from
- the **context cost** of carrying it (stays near-zero until used).

The concrete instance below — Agent Skills — makes the mechanic vivid with three loading levels:

| Level | Loaded when | Rough token cost | Content |
|---|---|---|---|
| **1 — Metadata** | Always (at startup) | **~100 tokens per skill** | `name` + `description` only |
| **2 — Instructions** | When a request matches the description | Under ~5k tokens | the `SKILL.md` body |
| **3 — Resources** | As needed, if the body references them | none until accessed | bundled files (read into context) and scripts (**run via bash — only their *output* enters context, never the code**) |

Two consequences make this more than a tidy-prompt trick:

- **You can attach dozens of capabilities for almost nothing.** At ~100 tokens each, a whole library of skills is a rounding error until one is actually triggered. Breadth stops fighting the context budget.
- **Bundled code buys determinism.** A skill can ship a *validated* script the agent runs instead of re-deriving fragile logic in-context every time — and because only the script's output returns, a 400-line utility costs zero context. This is the tool-design instinct from `02` taken one level further: don't make the model *do* what a correct function can do.

> **⭐ Key takeaway —** progressive disclosure is the durable idea; `SKILL.md` is just today's concrete spelling of it. If you internalize "advertise cheaply, load on demand, push determinism into bundled code," you'll recognize the same pattern under whatever a given agent platform calls it.

## The concrete instance: an Agent Skill

An **Agent Skill** is the packaging of that pattern that's now in wide use. At its simplest it's *a folder with a `SKILL.md` file* — YAML frontmatter (the cheap label) plus a Markdown body (the instructions), optionally alongside bundled scripts, references, and templates:

```
pdf-processing/
├── SKILL.md          # required: frontmatter (label) + Markdown body (instructions)
├── scripts/          # optional: code the agent runs via bash (output returns, not the code)
├── references/       # optional: docs the agent reads only when the body points to them
└── assets/           # optional: templates, schemas, example files
```

A minimal, portable `SKILL.md` looks like this — **reference only; there's nothing to run here:**

```markdown
---
name: pdf-processing
description: Extract text and tables from PDFs, fill forms, and merge documents.
  Use when the user mentions PDFs, forms, or document extraction.
---

# PDF Processing

## Instructions
1. To extract text, open the PDF with `pdfplumber` and read `page.extract_text()`.
2. For form filling, follow the steps in [FORMS.md](references/FORMS.md).
3. To merge documents, run `scripts/merge.py <in1> <in2> <out>`.

## Examples
- "Pull the text out of this contract and summarize it."
- "Fill in this application form with the values I gave you."
```

The whole design turns on **two required fields**, and the second is the one people underrate:

- **`name`** — a short identifier (lowercase, hyphens).
- **`description`** — the string the model matches a request against to decide whether to trigger the skill. It must say **both what the skill does *and* when to use it.** A vague description is the #1 reason a skill silently never fires — this is the same "write the schema for the model, not for yourself" discipline from [`02-tool-design`](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/02-tool-design.ipynb), applied to the label instead of the tool.

The portable open spec allows a few more optional fields (`license`, `compatibility`, `metadata`, and `allowed-tools` — the tools a skill may use without a per-call permission prompt). There is deliberately **no `version` field** — versioning lives outside the file (git, or a platform's version API). Individual agent hosts (e.g. Claude Code) add their own extra frontmatter keys, but those are host-specific extensions, not part of the portable core.

> **⚠️ Production reality —** the same `SKILL.md` is *not* magically shared across surfaces. A skill you add to a chat app, one registered via an API, and one sitting in a repo's `.claude/skills/` are separate installs in separate places. "Write once" refers to the *format*, not automatic sync — know where a given skill actually lives before you promise it's available.

## The trio that completes the agent story

You've now met three complementary layers. The clean mental model — and a great interview answer — is that they answer different questions:

| Layer | Answers | What it is | Context cost |
|---|---|---|---|
| **Tools** (`01`–`03`, `02-tool-design`) | *What can the agent **do**?* | individual function + schema the model can call | schema is **always** in context |
| **MCP** (`04-mcp`) | *What external systems can the agent **reach**?* | an open protocol that connects the agent to servers, data, APIs | per-connected-tool schema in context |
| **Skills** (this notebook) | *How should the agent **do a task well**?* | a packaged, progressively-disclosed folder of procedure + resources | **~100 tokens** until triggered |

They compose rather than compete. A useful way to hold it: **MCP connects the agent to external systems; Skills package the procedural know-how for using them well.** A skill's instructions can even orchestrate MCP-provided tools, and a skill can bundle its own deterministic code to run as a tool. Tools are the verbs, MCP is the wiring to the outside world, Skills are the playbooks.

> **🔵 Signal —** an engineer who can place all three on this grid — and say *why* Skills are metadata-gated while tool schemas are always-resident — is demonstrating that they understand the context budget as an architectural constraint, not just a billing line. That framing lands in interviews.

## When an FDE reaches for a Skill (and when not)

**Reach for a skill when:**
- The agent needs **repeatable procedural know-how** — a runbook, a house format, a multi-step workflow — that's too detailed to live in the system prompt but is needed only *sometimes*.
- You want to **bundle a validated script or reference file** so the agent runs correct code / reads exact specs instead of improvising them each time.
- You're building **many capabilities** and the always-resident prompt is getting expensive or muddled — progressive disclosure is exactly the pressure valve.
- You want the capability **portable across agent hosts** — the format is an open standard adopted well beyond its origin (see below).

**Skip it (keep a plain prompt or tool) when:**
- The instruction is **short and always relevant** — if it belongs in every turn anyway, a system-prompt line is simpler than a skill.
- You need **one function call**, not a procedure — that's a tool (`02`), not a skill.
- You're **learning the loop** — which is why this repo builds tools by hand first. You can't judge whether a packaging layer helps until you know what it's packaging.

> **💡 Why the origin story matters —** the Agent Skill format began at Anthropic but was released as an **open standard** (agentskills.io) and is now consumed by multiple agent products (OpenAI's Codex, Google's Gemini CLI, editor assistants, and more). That's the tell that you're learning a *durable pattern*, not a single vendor's feature — the same reason this repo teaches raw APIs and named patterns over any one wrapper.

The judgment is the repo's usual one: **know the raw pattern (context is a budget; advertise cheaply, load on demand), and adopt the standard when the breadth of capability — not the demo — justifies it.**

## Security note — bundled instructions and code are a trust surface

The same warning that closed the MCP notebook applies here, sharpened. A skill is *instructions the model will follow* plus, often, *code the agent will execute* — so an installed skill is as trusted as the person who wrote it. A skill pulled from a public catalog can carry indirect prompt-injection in its body, or a script that does more than it claims. "It's just a skill" is not a safety argument.

Treat third-party skills the way [`07-security`](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/07-security/01-prompt-injection-and-trust.ipynb) says to treat any untrusted input: read before you install, scope `allowed-tools` tightly, prefer sandboxed execution, and human-gate consequential actions. Progressive disclosure makes capability cheap to *add* — which makes the discipline of *what you add* more important, not less.

**Further reading:** the [Agent Skills open standard](https://agentskills.io) (spec + the list of adopting clients) and Anthropic's [engineering write-up on Agent Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills).

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Keep always-relevant, short instructions in the prompt; package *occasional, detailed* know-how as a skill | Stuff every playbook into the system prompt and pay for all of it every turn |
| Write the `description` for the model — what it does **and when to use it** | A vague `description`, then wonder why the skill never triggers |
| Push fragile logic into a bundled, validated script (only its output costs context) | Make the model re-derive the same brittle procedure in-context each time |
| Treat installed skills as trusted code; review third-party ones (see `07`) | Install a public skill and assume its body/scripts are safe |
| Reach for a skill when *breadth* strains the context budget | Reach for the packaging layer before you understand the tool loop it sits on |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises (no code — reasoning)

1. **Do the arithmetic.** Suppose you have 12 detailed playbooks, ~2k tokens each, and a typical task uses one. Compare the per-turn context cost of (a) all 12 in the system prompt vs (b) 12 skills at ~100 tokens of metadata each plus the one triggered body. What does that ratio do to cost and latency at 100k calls/day?
2. **Write a description that fires.** Pick a task from your own work and draft a skill `description` (2 sentences max) that states *what it does and when to use it* well enough that a model would trigger it for the right request and ignore it otherwise. Then write a bad version and say why it would misfire.
3. **Place it on the grid.** Take one capability from your capstone (section 10) and decide: is it a Tool, an MCP connection, or a Skill — or a Skill that *uses* a tool? Justify it in two sentences to an interviewer.
4. **Script vs prompt.** Name one piece of logic in your project that's currently "ask the model to do it" and would be more reliable as a bundled script the skill runs. What does moving it buy you in determinism *and* in context cost?
5. **Threat-model a public skill.** You install a third-party "invoice-formatter" skill that ships a script. List two ways it could harm you and the mitigation for each. (Revisit after [`07-security`](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/07-security/01-prompt-injection-and-trust.ipynb).)